# DirectQuery Semantic Model - Best Practices Check

This notebook inspects a Power BI / Fabric semantic model that uses **DirectQuery** and validates it against the best-practice checklist:

1. Star schema fundamentals (fact/dimension shape)
2. All Dimension tables in **Dual** storage mode
3. **Assume Referential Integrity** enabled on relationships
4. Relationship columns are **Integer** data type
5. Auto Aggregations enabled
6. Query parallelism / `MaxParallelismPerQuery` configured
7. SKU-appropriate DQ concurrent connections / parallelism

> Run this notebook inside a **Microsoft Fabric** workspace (the `sempy` / Semantic Link library is pre-installed there). It can also be run locally if you install `semantic-link` and authenticate.

References: [DirectQuery guidance](https://learn.microsoft.com/power-bi/guidance/directquery-model-guidance), [Storage modes](https://learn.microsoft.com/power-bi/transform-model/desktop-storage-mode), [Assume RI](https://learn.microsoft.com/power-bi/connect-data/desktop-assume-referential-integrity), [Auto Agg](https://learn.microsoft.com/power-bi/enterprise/aggregations-auto-configure).

## 1. Install & import Semantic Link
`sempy` is already available in Fabric notebooks. Uncomment the `%pip install` line if running elsewhere.

In [ ]:
# %pip install semantic-link --quiet

import sempy.fabric as fabric
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

## 2. Configure the target semantic model
Set `WORKSPACE` and `DATASET` to the workspace + semantic model you want to audit. Leave `WORKSPACE = None` to use the current workspace.

In [ ]:
WORKSPACE = None                 # e.g. "Sales Analytics" or a workspace GUID
DATASET   = "<Your Semantic Model Name>"

# List available datasets in the workspace for convenience
fabric.list_datasets(workspace=WORKSPACE)

## 3. Pull model metadata
Retrieve tables, columns, relationships and partitions from the semantic model.

In [ ]:
tables        = fabric.list_tables(DATASET, workspace=WORKSPACE, additional_xmla_properties=["Description"])
columns       = fabric.list_columns(DATASET, workspace=WORKSPACE, additional_xmla_properties=["SourceColumn"])
relationships = fabric.list_relationships(DATASET, workspace=WORKSPACE)
partitions    = fabric.list_partitions(DATASET, workspace=WORKSPACE)

print(f"Tables:        {len(tables)}")
print(f"Columns:       {len(columns)}")
print(f"Relationships: {len(relationships)}")
print(f"Partitions:    {len(partitions)}")
partitions.head()

## 4. Identify DirectQuery usage & storage modes
Each partition exposes a `Mode` (`Import`, `DirectQuery`, `Dual`). A table is considered DirectQuery if any of its partitions use DQ.

In [ ]:
mode_col = "Mode" if "Mode" in partitions.columns else "Storage Mode"
table_col = "Table Name" if "Table Name" in partitions.columns else "TableName"

storage = (partitions.groupby(table_col)[mode_col]
                     .agg(lambda s: ",".join(sorted(set(s))))
                     .reset_index()
                     .rename(columns={table_col: "Table", mode_col: "StorageMode"}))

is_dq_model = storage["StorageMode"].str.contains("DirectQuery").any()
print("Model uses DirectQuery:", is_dq_model)
storage

## 5. Classify fact vs dimension tables
Heuristic: a **fact** table participates on the *many* side of one or more relationships; a **dimension** is on the *one* side. This lets us evaluate star-schema shape and required storage modes.

In [ ]:
from_col = "From Table"
to_col   = "To Table"

fact_tables      = set(relationships[from_col].unique())   # many side
dimension_tables = set(relationships[to_col].unique())     # one side
# A table that is only on the one-side is a pure dimension
pure_dimensions  = dimension_tables - fact_tables
pure_facts       = fact_tables - dimension_tables

classification = pd.DataFrame({
    "Table": sorted(set(storage["Table"])),
})
classification["Role"] = classification["Table"].apply(
    lambda t: "Fact" if t in pure_facts else ("Dimension" if t in pure_dimensions else ("Bridge/Other" if t in fact_tables else "Standalone"))
)
classification = classification.merge(storage, on="Table", how="left")
classification

## Check 1 — Star schema fundamentals
Flag snowflaking (dimensions joined to other dimensions) and standalone tables.

In [ ]:
# Snowflake: a table on the 'many' side that is itself a dimension of another table
snowflake_rels = relationships[
    relationships[from_col].isin(pure_dimensions | dimension_tables) &
    ~relationships[from_col].isin(pure_facts)
]

standalone = classification[classification["Role"] == "Standalone"]["Table"].tolist()

print("Snowflaked relationships (dimension -> dimension):", len(snowflake_rels))
display(snowflake_rels)
print("Standalone tables (no relationships):", standalone)

## Check 2 — All dimension tables must be in **Dual** mode
When the fact is DirectQuery, dimensions should be Dual so slicers / filter cardinality can resolve locally.

In [ ]:
dim_modes = classification[classification["Role"] == "Dimension"].copy()
dim_modes["DualMode_OK"] = dim_modes["StorageMode"].eq("Dual")

not_dual = dim_modes[~dim_modes["DualMode_OK"]]
print(f"Dimensions NOT in Dual mode: {len(not_dual)}")
not_dual

## Check 3 — Assume Referential Integrity on every relationship
The `RelyOnReferentialIntegrity` flag (a.k.a. Assume RI) enables inner joins on the source and is critical for DQ performance.

In [ ]:
ri_col = next((c for c in relationships.columns if c.lower().replace(" ", "") in
               ("relyonreferentialintegrity", "assumereferentialintegrity")), None)

if ri_col is None:
    # Fetch the property explicitly
    relationships = fabric.list_relationships(
        DATASET, workspace=WORKSPACE,
        additional_xmla_properties=["RelyOnReferentialIntegrity"])
    ri_col = "RelyOnReferentialIntegrity"

missing_ri = relationships[relationships[ri_col].astype(str).str.lower().isin(["false", "0", "nan"])]
print(f"Relationships missing Assume RI: {len(missing_ri)} / {len(relationships)}")
missing_ri[[from_col, "From Column", to_col, "To Column", ri_col]]

## Check 4 — Relationship columns must be Integer
String / GUID / double-typed join keys kill DQ performance.

In [ ]:
cols = columns.rename(columns={"Table Name": "Table", "Column Name": "Column"})

def lookup_type(table, column):
    m = cols[(cols["Table"] == table) & (cols["Column"] == column)]
    return m.iloc[0]["Data Type"] if not m.empty else None

rel_types = relationships.copy()
rel_types["FromType"] = rel_types.apply(lambda r: lookup_type(r[from_col], r["From Column"]), axis=1)
rel_types["ToType"]   = rel_types.apply(lambda r: lookup_type(r[to_col],   r["To Column"]),   axis=1)

INT_TYPES = {"Int64", "Integer", "Whole Number", "Int32"}
bad_types = rel_types[~(rel_types["FromType"].isin(INT_TYPES) & rel_types["ToType"].isin(INT_TYPES))]
print(f"Relationships with non-integer keys: {len(bad_types)}")
bad_types[[from_col, "From Column", "FromType", to_col, "To Column", "ToType"]]

## Check 5 — Auto Aggregations

Auto-Aggregation training is a **service-side dataset setting** and is *not* a direct property on the TOM `Model`. We detect it two ways:

1. **TOM side** — look for aggregation tables (hidden tables with columns that have `AlternateOf` set). When auto-agg has trained at least once, these objects exist.
2. **Service side** — call the Power BI REST API (`/datasets/{id}/queryScaleOutSettings` and the dataset settings endpoint) via `sempy`'s `PowerBIRestClient` to read the actual training toggle.

If both signals are absent the feature is considered off; if either is present it is on.

In [ ]:
# ---------------------------------------------------------------
# (A) TOM-side detection: look for aggregation artifacts in the model
# ---------------------------------------------------------------
auto_agg_tom = False
agg_columns  = []
max_parallel = None
default_mode = None

try:
    tom = fabric.create_tom_server(workspace=WORKSPACE)
    db = (tom.Databases.GetByName(DATASET)
          if hasattr(tom.Databases, "GetByName") else tom.Databases[DATASET])
    model = db.Model

    max_parallel = getattr(model, "MaxParallelismPerQuery", None)
    default_mode = getattr(model, "DefaultMode", None)

    # Auto-agg training creates aggregation columns bound via AlternateOf.
    for t in model.Tables:
        for c in t.Columns:
            alt = getattr(c, "AlternateOf", None)
            if alt is not None:
                agg_columns.append((t.Name, c.Name, str(getattr(alt, "Summarization", "")),
                                    str(getattr(alt, "BaseColumn", "") or getattr(alt, "BaseTable", ""))))
    auto_agg_tom = len(agg_columns) > 0

    print("DefaultMode:            ", default_mode)
    print("MaxParallelismPerQuery: ", max_parallel)
    print(f"Aggregation columns found in model: {len(agg_columns)}")
    if agg_columns:
        display(pd.DataFrame(agg_columns, columns=["Table", "Column", "Summarization", "BaseObject"]))
except Exception as ex:
    print("TOM access not available:", ex)

# ---------------------------------------------------------------
# (B) Service-side detection: Power BI REST API
# ---------------------------------------------------------------
auto_agg_service = None
try:
    client = fabric.PowerBIRestClient()
    ds_id  = fabric.resolve_dataset_id(DATASET, workspace=WORKSPACE)
    ws_id  = fabric.resolve_workspace_id(WORKSPACE) if WORKSPACE else None

    # Dataset settings that surface auto-aggregation training
    url = (f"v1.0/myorg/groups/{ws_id}/datasets/{ds_id}"
           if ws_id else f"v1.0/myorg/datasets/{ds_id}")
    resp = client.get(url).json()

    # The property is exposed as "autoSyncReadOnlyReplicas" historically,
    # and as "queryScaleOutSettings" / "isAutomaticAggregationsEnabled" in newer APIs.
    for key in ("isAutomaticAggregationsEnabled",
                "autoAggregationsEnabled",
                "queryScaleOutSettings"):
        if key in resp:
            auto_agg_service = resp[key]
            print(f"REST property `{key}`:", auto_agg_service)
            break
    else:
        print("Auto-agg property not present in REST response. Full settings keys:",
              list(resp.keys()))
except Exception as ex:
    print("REST API check skipped:", ex)

# ---------------------------------------------------------------
# Final verdict
# ---------------------------------------------------------------
auto_agg_enabled = bool(auto_agg_tom) or bool(auto_agg_service)
print("\nAuto Aggregations enabled:", auto_agg_enabled,
      f"(tom={auto_agg_tom}, service={auto_agg_service})")

## Check 6 — Query parallelism / MaxParallelismPerQuery
For F64+ capacities, raising `Model.MaxParallelismPerQuery` can improve DQ performance. Compare against the SKU table below.

In [ ]:
sku_limits = pd.DataFrame([
    ("F2",    3,   5,  "1"),
    ("F4",    3,   5,  "1"),
    ("F8",    3,  10,  "1"),
    ("F16",   5,  10,  "1"),
    ("F32",  10,  10,  "1"),
    ("F64",  25,  50,  "4-8"),
    ("F128", 50,  75,  "6-12"),
    ("F256", 100, 100, "8-16"),
    ("F512", 200, 200, "10-20"),
    ("F1024",400, 200, "12-24"),
    ("F2048",400, 200, "12-24"),
], columns=["SKU", "MaxMemoryGB", "MaxConcurrentDQConnections", "MaxDQParallelism"])
sku_limits

## Check 7 — Per-data-source `MaxConnections` (model-level DQ connection cap)

`MaxConcurrentDQConnections` (per-SKU) is a *capacity* limit and can't be changed. But each model data source has a tunable **`MaxConnections`** property (TOM: `DataSource.MaxConnections`, default `10`) that limits concurrent DQ connections opened by the engine against that source. It should be ≤ the SKU's `MaxConcurrentDQConnections` to avoid queueing.

This cell reads `MaxConnections` for every data source in the model via TOM and compares against an optional target SKU.

In [ ]:
# Target SKU of the capacity that hosts the workspace (used only for the comparison below).
TARGET_SKU = "F64"   # <- change to match your capacity: F2, F4, F8, F16, F32, F64, F128, F256, F512, F1024, F2048

ds_rows = []
try:
    # Reuse `model` from Check 5 if present, otherwise re-open.
    if "model" not in dir() or model is None:
        tom = fabric.create_tom_server(workspace=WORKSPACE)
        db = (tom.Databases.GetByName(DATASET)
              if hasattr(tom.Databases, "GetByName") else tom.Databases[DATASET])
        model = db.Model

    # 1) Classic (legacy) DataSource objects
    for ds in getattr(model, "DataSources", []) or []:
        ds_rows.append({
            "DataSource":     ds.Name,
            "Type":            str(getattr(ds, "Type", "Provider")),
            "MaxConnections":  getattr(ds, "MaxConnections", None),
            "ConnectionDetails": str(getattr(ds, "ConnectionString", "") or getattr(ds, "Account", "")),
        })

    # 2) Modern (Power Query / structured) sources live in partition M expressions.
    #    They don't expose MaxConnections individually; the model-level default (10) applies
    #    unless overridden via TOM scripting.
    pq_partitions = []
    for t in model.Tables:
        for p in t.Partitions:
            src = getattr(p, "Source", None)
            src_type = type(src).__name__ if src is not None else ""
            if "MPartitionSource" in src_type or "M" == src_type or "QueryPartitionSource" in src_type:
                pq_partitions.append((t.Name, p.Name, src_type))

    if not ds_rows and pq_partitions:
        print("Model uses Power Query / structured sources — no per-datasource "
              "`MaxConnections` object is exposed.")
        print(f"Power Query partitions: {len(pq_partitions)}")
        print("Engine default MaxConnections = 10 per source unless overridden by TOM scripting.")

except Exception as ex:
    print("TOM read failed:", ex)

ds_df = pd.DataFrame(ds_rows)
display(ds_df if not ds_df.empty else pd.DataFrame({"info": ["no legacy DataSource objects found"]}))

# Compare against the SKU's MaxConcurrentDQConnections
sku_row = sku_limits.loc[sku_limits["SKU"] == TARGET_SKU]
if not sku_row.empty and not ds_df.empty and "MaxConnections" in ds_df:
    sku_cap = int(sku_row.iloc[0]["MaxConcurrentDQConnections"])
    ds_df["SKU_Cap"] = sku_cap
    ds_df["OK"]      = ds_df["MaxConnections"].fillna(10).astype(int) <= sku_cap
    print(f"\nSKU {TARGET_SKU} cap = {sku_cap} concurrent DQ connections per semantic model")
    display(ds_df[["DataSource", "MaxConnections", "SKU_Cap", "OK"]])
    max_connections_ok = bool(ds_df["OK"].all())
else:
    max_connections_ok = None
    print("Could not evaluate MaxConnections vs SKU cap.")

## 6. Consolidated report

In [ ]:
report = []

report.append(("Model uses DirectQuery",
               "INFO", f"{is_dq_model}"))

report.append(("Star schema - snowflaked relationships",
               "PASS" if snowflake_rels.empty else "WARN",
               f"{len(snowflake_rels)} found"))

report.append(("Star schema - standalone tables",
               "PASS" if not standalone else "WARN",
               ", ".join(standalone) or "none"))

report.append(("Dimensions in Dual mode",
               "PASS" if not_dual.empty else "FAIL",
               f"{len(not_dual)} dimension(s) not Dual"))

report.append(("Assume Referential Integrity on all relationships",
               "PASS" if missing_ri.empty else "FAIL",
               f"{len(missing_ri)} relationship(s) missing RI"))

report.append(("Relationship keys are Integer",
               "PASS" if bad_types.empty else "FAIL",
               f"{len(bad_types)} relationship(s) use non-integer keys"))

report.append(("Auto Aggregations enabled",
               "PASS" if auto_agg_enabled else "WARN",
               f"tom={auto_agg_tom}, service={auto_agg_service}"))

report.append(("MaxParallelismPerQuery configured",
               "INFO", f"{max_parallel}"))

report.append(("DataSource MaxConnections <= SKU cap",
               "PASS" if max_connections_ok else ("WARN" if max_connections_ok is None else "FAIL"),
               f"SKU={TARGET_SKU}, sources={len(ds_df) if 'ds_df' in dir() else 0}"))

pd.DataFrame(report, columns=["Check", "Status", "Detail"])

## 7. Remediation hints
- **Dimensions not Dual**: open the model in Power BI Desktop -> select the table -> *Advanced properties* -> **Storage mode = Dual**.
- **Assume RI missing**: edit the relationship -> check **Assume referential integrity**.
- **Non-integer keys**: replace string/GUID keys with surrogate `Int64` keys in Power Query or the source.
- **Apply slicer button**: File -> Options -> Query reduction -> *Add an Apply button to each slicer*.
- **Auto Agg**: Semantic model settings -> *Aggregations training* -> On.
- **Parallelism**: use TOM scripting to set `Model.MaxParallelismPerQuery` (see the SKU table).